# LBT GCS Plotting

Sandbox for extracting LBT guider control system (GCS) telemetry data from HDF5 files and making plots of
GCS performance (seeing and photometry) for a night of LBT observing.

Note: the GCS files only contain info for the MODS and LUCI guiders

This sandbox starts by learning how to deconstruct the contents of the HDF files.


In [5]:
%matplotlib inline

import math
import h5py as hdf
import numpy as np
import glob

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, LogLocator, NullFormatter

# astropy time 

from astropy.time import Time

# throttle nuisance warnings

import warnings
warnings.filterwarnings('ignore',category=UserWarning, append=True)
warnings.filterwarnings('ignore',category=RuntimeWarning, append=True)

## Standard plot setup

Basic setup for the various plots we'll make, this is the common configuration for each plot.

In [6]:
# Plot width and height in pixels

plotHeight = 3000
plotWidth = 4000

# Font and line weight defaults for axes

lwidth = 0.5
matplotlib.rcParams.update({'font.size':10})
matplotlib.rc('axes',linewidth=lwidth)

# LaTeX will be used throughout for markup of symbols

plt.rc('text', usetex=True)
plt.rc('font', **{'family':'serif','serif':['Times-Roman'],'weight':'bold','size':'10'})
plt.rcParams['xtick.major.pad']='5'
plt.rcParams['ytick.major.pad']='5'
plt.rcParams['axes.labelpad'] = '5'

# plot resolution and window

dpi = 600
wDisp = plotWidth
hDisp = plotHeight
wInches = float(wDisp)/float(dpi)
hInches = float(hDisp)/float(dpi)

## Plot GCS image quality and photometry

Open/sniff/close HDF5 files with the GCS telemetry, make a 2-panel (over/under) plot
 * Guide Star image quality (FWHM and ellipticity)
 * Guide star instrumental magnitude $r_{inst}=27.0-2.5\log_{10}(cts/sec)$
 
The instrumental magnitude zero point is mostly a guess that puts it in about the right range. We should revisit
this during good transparency conditions and put in a number consistent with the R magnitudes of the actual 
guide stars.
 
### Open and read the HDF5 telemetry files

We use these data elements:
 * time_stamp - MJD in integer microseconds
 * avgseeing - average seeing FWHM in arcseconds
 * sxellip - SExtractor measured image ellipticity
 * sxflux - SExtractor measured image flux above background in counts
 * exptime - guide camera exposure time in milliseconds
 
Other data might be used, this gets us going.

Use glob to get the GCS HDF5 files for a given CCYYMMDD date tag.


In [7]:
obsDate = '20260510'

haveSX = False
haveDX = False

leftList = glob.glob(f'GCS/{obsDate}*.gcsl.guiding.h5')
if len(leftList)==1:
    leftFile = leftList[0]
    haveSX = True
else:
    leftFile = 'gcsl.h5'
    
rightList = glob.glob(f'GCS/{obsDate}*.gcsr.guiding.h5')
if len(rightList)==1:
    rightFile = rightList[0]
    haveDX = True
else:
    rightFile = 'gcsr.h5'

# open the HDF files

if haveSX:
    gcsl = hdf.File(leftFile,'r') # read only
if haveDX:
    gcsr = hdf.File(rightFile,'r')

if haveSX:
    sxDS = list(gcsl.keys())[0]
    sxData = gcsl[sxDS]

    for datum in ['time_stamp','avgseeing','sxellip','sxflux','exptime']:
        print(f'\n{datum}:')
        for key in ['Description','Units']:
            print(f"  {key}: {sxData.attrs[key][datum]}")


    sxMJD = sxData['time_stamp']*1.0e-6/86400.0 # convert MJD in microseconds to days
    mjd0 = int(sxMJD[0])
    sxHour = 24.0*(sxMJD-mjd0)

    sxFWHM = sxData['avgseeing']
    sxEll = sxData['sxellip']
    sxFlux = sxData['sxflux']
    sxExpT = 1.0e-3*sxData['exptime']
    sxMag = 27.0 - 2.5*np.log10(sxFlux/sxExpT)

if haveDX:
    dxDS = list(gcsr.keys())[0]
    dxData = gcsr[dxDS]

    dxMJD = dxData['time_stamp']*1.0e-6/86400.0 # convert MJD in microseconds to days
    if not haveSX:
        mjd0 = int(dxMJD[0])
    
    dxHour = 24.0*(dxMJD-mjd0)

    dxFWHM = dxData['avgseeing']
    dxEll = dxData['sxellip']
    dxFlux = dxData['sxflux']
    dxExpT = 1.0e-3*dxData['exptime']
    dxMag = 27.0 - 2.5*np.log10(dxFlux/dxExpT)

# UTC date from the starting MJD

t = Time([mjd0], format='mjd', scale='utc')
utcDate = f"{t.to_value('iso',subfmt='date')[0]}"

# plotting limits

if not haveSX:
    minTime = np.min(dxHour)
    maxTime = np.max(dxHour)
    medFWHM = np.median(dxFWHM)
    minMag = np.nanmin(dxMag)
    maxMag = np.nanmax(dxMag)
elif not haveDX:
    minTime = np.min(sxHour)
    maxTime = np.max(sxHour)
    medFWHM = np.median(sxFWHM)
    minMag = np.nanmin(sxMag)
    maxMag = np.nanmax(sxMag)
else:
    minTime = np.min([np.min(sxHour),np.min(dxHour)])
    maxTime = np.max([np.max(sxHour),np.max(dxHour)])
    medFWHM = np.max([np.median(sxFWHM),np.median(dxFWHM)])
    minMag = np.min([np.nanmin(dxMag),np.nanmin(sxMag)])
    maxMag = np.min([np.nanmax(dxMag),np.nanmax(sxMag)])
    
dT = maxTime - minTime
tMin = np.floor(minTime) # minTime - 0.05*dT
tMax = np.ceil(maxTime+0.01*dT) # maxTime + 0.05*dT
#tMin = 7.5
#tMax = 7.5

fwMin = 0.0
fwMax = np.ceil(2.0*medFWHM)
print(f'fwMax={fwMax:.2f} arcsec')
#fwMax = 3.0

print(f'minMag={minMag:.2f}, maxMax={maxMag:.2f}')
dm = maxMag - minMag
mMin = np.ceil(maxMag + 0.05*dm)
mMax = np.floor(minMag - 0.05*dm)



time_stamp:
  Description: b'MJD UTC'
  Units: b'microsecond'

avgseeing:
  Description: b'Average FWHM'
  Units: b'arcsecond'

sxellip:
  Description: b'Sextractor measured ellipticity'
  Units: b'none'

sxflux:
  Description: b'Sextractor measured flux in counts'
  Units: b'none'

exptime:
  Description: b'Guide camera exposure time'
  Units: b'millisecond'
fwMax=3.00 arcsec
minMag=10.40, maxMax=14.21


## Make The Plot


In [8]:
# make a plot

fig,(ax1,ax2) = plt.subplots(2,1,figsize=(wInches,hInches),dpi=dpi)
fig.subplots_adjust(wspace=0, hspace=0.2)

# Top panel - image quality

ax1.tick_params('both',length=4,width=lwidth,which='major',direction='in',top='on',right='on')
ax1.tick_params('both',length=2,width=lwidth,which='minor',direction='in',top='on',right='on')

# Limits

ax1.set_ylabel(r'FWHM [arcsec] \& ellipticity')

ax1.xaxis.set_major_locator(MultipleLocator(1.0))
ax1.xaxis.set_minor_locator(MultipleLocator(0.25))
ax1.set_xlim(tMin,tMax)

ax1.yaxis.set_major_locator(MultipleLocator(0.5))
ax1.yaxis.set_minor_locator(MultipleLocator(0.1))
ax1.set_ylim(fwMin,fwMax)

if haveSX:
    ax1.plot(sxHour,sxFWHM,marker='o',mfc='black',ms=1,zorder=10,mew=0,lw=0,label='SX guider')
    ax1.plot(sxHour,sxEll,marker='o',color='black',ms=0.5,mew=0,zorder=10,lw=0,alpha=0.5)
if haveDX:
    ax1.plot(dxHour,dxFWHM,marker='o',mfc='green',ms=1,zorder=10,mew=0,lw=0,label='DX guider')
    ax1.plot(dxHour,dxEll,marker='o',color='green',ms=0.5,mew=0,zorder=10,lw=0,alpha=0.5)

ax1.set_title(rf'LBT AGw guide star data for {utcDate} UTC',fontsize=10)
ax1.legend(fontsize=8,markerscale=3)

# Bottom Panel - photometry

ax2.tick_params('both',length=4,width=lwidth,which='major',direction='in',top='on',right='on')
ax2.tick_params('both',length=2,width=lwidth,which='minor',direction='in',top='on',right='on')

# Limits

ax2.set_xlabel(rf'Time (UT Hours) for {utcDate} UTC')
ax2.set_ylabel(r'Instrumental R Mag')

ax2.xaxis.set_major_locator(MultipleLocator(1.0))
ax2.xaxis.set_minor_locator(MultipleLocator(0.25))
ax2.set_xlim(tMin,tMax)

ax2.yaxis.set_major_locator(MultipleLocator(1.0))
ax2.yaxis.set_minor_locator(MultipleLocator(0.2))
ax2.set_ylim(mMin,mMax)

if haveSX:
    ax2.plot(sxHour,sxMag,marker='o',mfc='black',ms=1,zorder=10,mew=0,lw=0)
if haveDX:
    ax2.plot(dxHour,dxMag,marker='o',mfc='green',ms=1,zorder=10,mew=0,lw=0)

plt.plot()

# hardcopy

plotFile = f'{obsDate}_gcs.png'
plt.savefig(plotFile,bbox_inches='tight',facecolor='white')